# GeoLibre WASM quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geolibre-rust/blob/main/examples/geolibre_wasm.ipynb)

Run the [`geolibre-rust`](https://github.com/opengeos/geolibre-rust) geospatial
tool suite (the `whitebox_next_gen` tools plus GeoLibre's own) from Python. The
tools are a single WebAssembly (WASI) module executed in-process via `wasmtime`,
so there is **no native install, no GDAL, and no server**.

This notebook reads and processes real public datasets:

- raster DEM: `https://data.source.coop/giswqs/opengeos/dem.tif`
- vector buildings: `https://data.source.coop/giswqs/opengeos/las-vegas-buildings.geojson`


## Install

In [ ]:
%pip install -q "geolibre-wasm>=0.4.3"

## Setup

The tools run over an in-memory `/work` filesystem: you pass input files as
`bytes` keyed by name, and any new files come back in `result.files` keyed by
their `/work`-relative path. Here we fetch the sample datasets with `urllib`.

In [ ]:
import urllib.request
import geolibre_wasm as gl


def fetch(url):
    """Download a file to bytes (with a browser User-Agent for picky hosts)."""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r:
        return r.read()


print(len(gl.list_tools()), "tools available")  # first call downloads the runtime once


### Tip: pass a URL or path directly

Instead of downloading first, an `input` value can be an `http(s)` URL (fetched
for you) or a local file path. The download-once pattern above is more efficient
when you reuse the same file across several calls; the shortcut is handy for
one-off runs and works the same for raster and vector inputs.

In [ ]:
# Same result as the cells below, but geolibre_wasm fetches the URL for you:
res = gl.run_tool(
    "slope",
    args=["--input=/work/dem.tif", "--output=/work/slope.tif", "--units=degrees"],
    input={"dem.tif": "https://data.source.coop/giswqs/opengeos/dem.tif"},
)
print(res.exit_code, res.stdout[-1])

## Raster: a DEM (Cloud Optimized GeoTIFF)

In [ ]:
DEM_URL = "https://data.source.coop/giswqs/opengeos/dem.tif"
dem = fetch(DEM_URL)

# Slope (degrees) -> Cloud Optimized GeoTIFF
res = gl.run_tool(
    "slope",
    args=["--input=/work/dem.tif", "--output=/work/slope.tif", "--units=degrees"],
    input={"dem.tif": dem},
)
assert res.exit_code == 0, res.stdout
print(res.stdout[-1])
open("slope.tif", "wb").write(res.files["slope.tif"])  # save the COG


Render a colormapped PNG preview and show it inline:

In [ ]:
from IPython.display import Image

res = gl.run_tool(
    "render_raster_png",
    args=["--input=/work/dem.tif", "--colormap=terrain", "--output=/work/dem.png"],
    input={"dem.tif": dem},
)
Image(data=res.files["dem.png"])


Reproject (warp) the DEM to another CRS (here WGS84 / EPSG:4326):

In [ ]:
res = gl.run_tool(
    "reproject_raster",
    args=["--input=/work/dem.tif", "--epsg=4326", "--output=/work/dem_4326.tif"],
    input={"dem.tif": dem},
)
print(res.stdout[-1])  # reports src_epsg, dst_epsg, rows, cols


Slice the DEM into a Web Mercator XYZ PNG tile pyramid for web maps:

In [ ]:
res = gl.run_tool(
    "raster_to_tiles",
    args=["--input=/work/dem.tif", "--output_dir=/work/tiles",
          "--min_zoom=10", "--max_zoom=12", "--colormap=terrain"],
    input={"dem.tif": dem},
)
print(res.stdout[-1])
tiles = [k for k in res.files if k.endswith(".png")]
print(len(tiles), "tiles, e.g.", tiles[0])  # keys look like tiles/12/700/1635.png


## Vector: building footprints (GeoJSON, Shapefile, FlatGeobuf, GeoParquet, ...)

In [ ]:
BLDG_URL = "https://data.source.coop/giswqs/opengeos/las-vegas-buildings.geojson"
buildings = fetch(BLDG_URL)

# Convert to GeoParquet (Hilbert-sorted, bbox covering, ZSTD by default)
res = gl.run_tool(
    "write_geoparquet",
    args=["--input=/work/buildings.geojson", "--output=/work/buildings.parquet"],
    input={"buildings.geojson": buildings},
)
print(res.stdout[-1])  # feature_count, compression, hilbert_sorted
open("las-vegas-buildings.parquet", "wb").write(res.files["buildings.parquet"])


Add geometry attributes (area, perimeter, centroid, ...):

In [ ]:
res = gl.run_tool(
    "add_geometry_attributes",
    args=["--input=/work/buildings.geojson", "--area=true", "--perimeter=true",
          "--centroid=true", "--output=/work/buildings_attrs.geojson"],
    input={"buildings.geojson": buildings},
)
print(res.exit_code, list(res.files))


Read GeoParquet back to another vector format (driver from the extension):

In [ ]:
parquet = open("las-vegas-buildings.parquet", "rb").read()
res = gl.run_tool(
    "read_geoparquet",
    args=["--input=/work/in.parquet", "--output=/work/out.fgb"],  # FlatGeobuf
    input={"in.parquet": parquet},
)
print(res.stdout[-1])  # feature_count


## Next steps

- `gl.list_tools()` returns every tool id (740+).
- `gl.list_manifests()` returns each tool's parameters and provenance
  (`"source": "geolibre" | "whitebox"`).

```python
manifests = {m["id"]: m for m in gl.list_manifests()}
manifests["slope"]["params"]
```

Docs: https://github.com/opengeos/geolibre-rust
